# Pre-processamento de Imagens — Redimensionamento

Este notebook le as imagens da pasta Fotos-Micro no Google Drive,
redimensiona para 512x512 px e guarda numa nova pasta Fotos-Micro-512.

As imagens originais nao sao alteradas.

Formato de saida: JPEG com qualidade 95 — compativel com TensorFlow.

---

## Passo 1 - Ligar ao Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Drive ligado.')

## Passo 2 - Configuracao

In [ ]:
from pathlib import Path

# Pasta de entrada — pasta original com as imagens
INPUT_DIR  = Path('/content/drive/MyDrive/Fotos-Micro/Fotos-Micro')

# Pasta de saida — sera criada automaticamente
OUTPUT_DIR = Path('/content/drive/MyDrive/Fotos-Micro/Fotos-Micro-512')

# Tamanho alvo em pixeis
TARGET_SIZE = (512, 512)

# Qualidade JPEG (1-95, valores altos = melhor qualidade, ficheiro maior)
JPEG_QUALITY = 95

# Extensoes de imagem a processar
EXTENSIONS = {'.jpg', '.jpeg', '.png', '.tif', '.tiff'}

print(f'Pasta de entrada : {INPUT_DIR}')
print(f'Pasta de saida   : {OUTPUT_DIR}')
print(f'Tamanho alvo     : {TARGET_SIZE[0]}x{TARGET_SIZE[1]} px')
print(f'Qualidade JPEG   : {JPEG_QUALITY}')

if not INPUT_DIR.exists():
    print(f'\nERRO: pasta de entrada nao encontrada: {INPUT_DIR}')
else:
    species_folders = [f for f in INPUT_DIR.iterdir() if f.is_dir()]
    print(f'\n{len(species_folders)} pastas de especies encontradas.')

## Passo 3 - Instalar e importar bibliotecas

In [ ]:
from PIL import Image
import os

print('PIL (Pillow) disponivel.')
print(f'Versao: {Image.__version__}')

## Passo 4 - Redimensionar imagens

In [ ]:
from PIL import Image
from pathlib import Path

processed = 0
skipped   = 0
errors    = 0

for species_dir in sorted(INPUT_DIR.iterdir()):
    if not species_dir.is_dir():
        continue

    for larva_dir in sorted(species_dir.iterdir()):
        if not larva_dir.is_dir():
            continue

        for img_path in sorted(larva_dir.iterdir()):
            if img_path.suffix.lower() not in EXTENSIONS:
                continue

            # Mirror the folder structure in the output directory
            relative = img_path.relative_to(INPUT_DIR)
            out_path = OUTPUT_DIR / relative.parent / (relative.stem + '.jpg')
            out_path.parent.mkdir(parents=True, exist_ok=True)

            # Skip if already processed
            if out_path.exists():
                skipped += 1
                continue

            try:
                with Image.open(img_path) as img:
                    # Convert to RGB (handles TIFF, grayscale, RGBA)
                    img_rgb = img.convert('RGB')

                    # Resize using high-quality Lanczos resampling
                    img_resized = img_rgb.resize(TARGET_SIZE, Image.LANCZOS)

                    # Save as JPEG
                    img_resized.save(out_path, 'JPEG', quality=JPEG_QUALITY)
                    processed += 1

                    if processed % 50 == 0:
                        print(f'  {processed} images processed...')

            except Exception as e:
                print(f'  ERROR: {img_path.name} — {e}')
                errors += 1

print(f'\nDone.')
print(f'  Processed : {processed}')
print(f'  Skipped   : {skipped} (already existed)')
print(f'  Errors    : {errors}')
print(f'  Output    : {OUTPUT_DIR}')

## Passo 5 - Verificar resultado

In [ ]:
from PIL import Image
import random

# Count output images
all_output = list(OUTPUT_DIR.rglob('*.jpg'))
print(f'Total images in output folder: {len(all_output)}')

# Check a random sample
print('\nChecking 5 random images:')
sample = random.sample(all_output, min(5, len(all_output)))
for f in sample:
    with Image.open(f) as img:
        print(f'  {f.name:<50} {img.size}  {img.mode}')

# Compare file sizes
print('\nFile size comparison (first 3 images):')
for out_f in all_output[:3]:
    relative  = out_f.relative_to(OUTPUT_DIR)
    orig_path = INPUT_DIR / relative.parent / relative.stem
    for ext in EXTENSIONS:
        candidate = orig_path.with_suffix(ext)
        if candidate.exists():
            orig_size = candidate.stat().st_size / 1024
            new_size  = out_f.stat().st_size / 1024
            print(f'  {out_f.name:<45} {orig_size:>7.1f} KB -> {new_size:>7.1f} KB')
            break